# 08_tabular_baseline

Final tabular baseline on the full PaySim dataset.
This notebook trains and evaluates a baseline fraud detection model using the final clean PaySim splits.

In [11]:
from pathlib import Path
import json
import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
)

In [12]:
project_root = Path.cwd().parent
results_dir = project_root / "results" / "tabular_full_baseline"
results_dir.mkdir(parents=True, exist_ok=True)

print("Results dir:", results_dir)

Results dir: c:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\results\tabular_full_baseline


In [13]:
target_col = "isFraud"

categorical_features = ["type"]
numeric_features = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "isFlaggedFraud",
]

feature_cols = categorical_features + numeric_features

X_train = paysim_train[feature_cols].copy()
y_train = paysim_train[target_col].copy()

X_val = paysim_val[feature_cols].copy()
y_val = paysim_val[target_col].copy()

X_test = paysim_test[feature_cols].copy()
y_test = paysim_test[target_col].copy()

print("Features used:", feature_cols)
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

Features used: ['type', 'step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud']
X_train shape: (4453834, 8)
X_val shape: (636262, 8)
X_test shape: (1272524, 8)


In [14]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['step', 'amount', 'oldbalanceOrg',
                                  'newbalanceOrig', 'oldbalanceDest',
                                  'newbalanceDest', 'isFlaggedFraud']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['type'])])

In [15]:
tabular_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        solver="saga",
        class_weight="balanced",
        max_iter=300,
        random_state=42
    )),
])

tabular_model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['step', 'amount',
                                                   'oldbalanceOrg',
                                                   'newbalanceOrig',
                                                   'oldbalanceDest',
                                                   'newbalanceDest',
                                                   'isFlaggedFraud']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['type'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=300,
                                    random_state=42, solver='saga'))])

A logistic regression baseline was trained on the full PaySim dataset.

In [16]:
tabular_model.fit(X_train, y_train)
print("Training finished.")

Training finished.


c:\Users\Admin\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [17]:
def evaluate_model(model, X, y, split_name="split"):
    y_pred = model.predict(X)
    y_score = model.predict_proba(X)[:, 1]

    metrics = {
        "split": split_name,
        "accuracy": float(accuracy_score(y, y_pred)),
        "precision": float(precision_score(y, y_pred, zero_division=0)),
        "recall": float(recall_score(y, y_pred, zero_division=0)),
        "f1": float(f1_score(y, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y, y_score)),
        "pr_auc": float(average_precision_score(y, y_score)),
    }

    cm = confusion_matrix(y, y_pred)

    print(f"\n=== {split_name.upper()} ===")
    for k, v in metrics.items():
        if k != "split":
            print(f"{k}: {v:.6f}")

    print("\nConfusion matrix:")
    print(cm)

    print("\nClassification report:")
    print(classification_report(y, y_pred, digits=4, zero_division=0))

    return metrics, cm

In [18]:
val_metrics, val_cm = evaluate_model(tabular_model, X_val, y_val, split_name="validation")
test_metrics, test_cm = evaluate_model(tabular_model, X_test, y_test, split_name="test")


=== VALIDATION ===
accuracy: 0.935747
precision: 0.018185
recall: 0.920828
f1: 0.035665
roc_auc: 0.982775
pr_auc: 0.557451

Confusion matrix:
[[594624  40817]
 [    65    756]]

Classification report:
              precision    recall  f1-score   support

           0     0.9999    0.9358    0.9668    635441
           1     0.0182    0.9208    0.0357       821

    accuracy                         0.9357    636262
   macro avg     0.5090    0.9283    0.5012    636262
weighted avg     0.9986    0.9357    0.9656    636262


=== TEST ===
accuracy: 0.935916
precision: 0.018465
recall: 0.932441
f1: 0.036212
roc_auc: 0.985584
pr_auc: 0.565464

Confusion matrix:
[[1189443   81438]
 [    111    1532]]

Classification report:
              precision    recall  f1-score   support

           0     0.9999    0.9359    0.9669   1270881
           1     0.0185    0.9324    0.0362      1643

    accuracy                         0.9359   1272524
   macro avg     0.5092    0.9342    0.5015   1272524

In [19]:
joblib.dump(tabular_model, results_dir / "tabular_logreg_full_paysim.joblib")

with open(results_dir / "validation_metrics.json", "w") as f:
    json.dump(val_metrics, f, indent=4)

with open(results_dir / "test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=4)

val_cm_df = pd.DataFrame(val_cm, index=["true_0", "true_1"], columns=["pred_0", "pred_1"])
test_cm_df = pd.DataFrame(test_cm, index=["true_0", "true_1"], columns=["pred_0", "pred_1"])

val_cm_df.to_csv(results_dir / "validation_confusion_matrix.csv")
test_cm_df.to_csv(results_dir / "test_confusion_matrix.csv")

metrics_summary = pd.DataFrame([val_metrics, test_metrics])
metrics_summary.to_csv(results_dir / "metrics_summary.csv", index=False)

print("Saved all outputs to:", results_dir)
metrics_summary

Saved all outputs to: c:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\results\tabular_full_baseline


,split,accuracy,precision,recall,f1,roc_auc,pr_auc
0,validation,0.935747,0.018185,0.920828,0.035665,0.982775,0.557451
1,test,0.935916,0.018465,0.932441,0.036212,0.985584,0.565464


In [20]:
print(metrics_summary.to_markdown(index=False))

| split      |   accuracy |   precision |   recall |        f1 |   roc_auc |   pr_auc |
|:-----------|-----------:|------------:|---------:|----------:|----------:|---------:|
| validation |   0.935747 |   0.0181849 | 0.920828 | 0.0356654 |  0.982775 | 0.557451 |
| test       |   0.935916 |   0.0184645 | 0.932441 | 0.0362119 |  0.985584 | 0.565464 |


## Baseline summary

A logistic regression model was trained as the tabular baseline on the full PaySim dataset.

The saved outputs include:
- trained model
- validation metrics
- test metrics
- confusion matrices
- metrics summary

For this highly imbalanced dataset, the most important metrics are precision, recall, F1, and PR-AUC.

In [21]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

val_scores = tabular_model.predict_proba(X_val)[:, 1]

thresholds = np.arange(0.1, 1.0, 0.05)
rows = []

for thr in thresholds:
    y_pred_thr = (val_scores >= thr).astype(int)
    rows.append({
        "threshold": thr,
        "precision": precision_score(y_val, y_pred_thr, zero_division=0),
        "recall": recall_score(y_val, y_pred_thr, zero_division=0),
        "f1": f1_score(y_val, y_pred_thr, zero_division=0),
        "predicted_fraud": int(y_pred_thr.sum()),
    })

threshold_results = pd.DataFrame(rows)
threshold_results

,threshold,precision,recall,f1,predicted_fraud
0,0.10,0.003315,1.000000,0.006608,247663
1,0.15,0.003620,1.000000,0.007215,226767
2,0.20,0.004106,1.000000,0.008179,199947
3,0.25,0.004840,0.997564,0.009632,169229
4,0.30,0.005925,0.989038,0.011779,137055
5,0.35,0.007459,0.969549,0.014805,106713
6,0.40,0.009871,0.957369,0.019540,79629
7,0.45,0.013378,0.941535,0.026381,57782
8,0.50,0.018185,0.920828,0.035665,41573
9,0.55,0.024283,0.896468,0.047286,30309


In [22]:
print(threshold_results.to_markdown(index=False))


|   threshold |   precision |   recall |         f1 |   predicted_fraud |
|------------:|------------:|---------:|-----------:|------------------:|
|        0.1  |  0.00331499 | 1        | 0.00660807 |            247663 |
|        0.15 |  0.00362046 | 1        | 0.00721479 |            226767 |
|        0.2  |  0.00410609 | 1        | 0.00817859 |            199947 |
|        0.25 |  0.0048396  | 0.997564 | 0.00963246 |            169229 |
|        0.3  |  0.00592463 | 0.989038 | 0.0117787  |            137055 |
|        0.35 |  0.00745926 | 0.969549 | 0.0148046  |            106713 |
|        0.4  |  0.00987078 | 0.957369 | 0.0195401  |             79629 |
|        0.45 |  0.0133779  | 0.941535 | 0.0263809  |             57782 |
|        0.5  |  0.0181849  | 0.920828 | 0.0356654  |             41573 |
|        0.55 |  0.0242832  | 0.896468 | 0.0472856  |             30309 |
|        0.6  |  0.0315319  | 0.859927 | 0.0608332  |             22390 |
|        0.65 |  0.0421279  | 0.834348

In [23]:
support_pos = int(y_val.sum())
support_neg = int((y_val == 0).sum())

cost_rows = []

for _, row in threshold_results.iterrows():
    thr = row["threshold"]
    recall = row["recall"]
    predicted_fraud = int(row["predicted_fraud"])

    tp = round(recall * support_pos)
    fn = support_pos - tp
    fp = predicted_fraud - tp
    tn = support_neg - fp

    expected_cost = fn * 50000 + fp * 500

    cost_rows.append({
        "threshold": thr,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": row["precision"],
        "recall": recall,
        "f1": row["f1"],
        "predicted_fraud": predicted_fraud,
        "expected_cost": expected_cost,
    })

cost_df = pd.DataFrame(cost_rows).sort_values("expected_cost")
cost_df

,threshold,tp,fp,fn,tn,precision,recall,f1,predicted_fraud,expected_cost
12,0.70,657,10666,164,624775,0.058023,0.800244,0.108202,11323,13533000
13,0.75,613,6675,208,628766,0.084111,0.746650,0.151190,7288,13737500
14,0.80,580,3766,241,631675,0.133456,0.706456,0.224502,4346,13933000
11,0.65,685,15575,136,619866,0.042128,0.834348,0.080206,16260,14587500
15,0.85,536,1850,285,633591,0.224644,0.652862,0.334269,2386,15175000
10,0.60,706,21684,115,613757,0.031532,0.859927,0.060833,22390,16592000
16,0.90,478,750,343,634691,0.389251,0.582217,0.466569,1228,17525000
9,0.55,736,29573,85,605868,0.024283,0.896468,0.047286,30309,19036500
17,0.95,395,177,426,635264,0.690559,0.481121,0.567121,572,21388500
8,0.50,756,40817,65,594624,0.018185,0.920828,0.035665,41573,23658500


In [24]:
print(cost_df.to_markdown(index=False))


|   threshold |   tp |     fp |   fn |     tn |   precision |   recall |         f1 |   predicted_fraud |   expected_cost |
|------------:|-----:|-------:|-----:|-------:|------------:|---------:|-----------:|------------------:|----------------:|
|        0.7  |  657 |  10666 |  164 | 624775 |  0.0580235  | 0.800244 | 0.108202   |             11323 |     1.3533e+07  |
|        0.75 |  613 |   6675 |  208 | 628766 |  0.0841109  | 0.74665  | 0.15119    |              7288 |     1.37375e+07 |
|        0.8  |  580 |   3766 |  241 | 631675 |  0.133456   | 0.706456 | 0.224502   |              4346 |     1.3933e+07  |
|        0.65 |  685 |  15575 |  136 | 619866 |  0.0421279  | 0.834348 | 0.0802061  |             16260 |     1.45875e+07 |
|        0.85 |  536 |   1850 |  285 | 633591 |  0.224644   | 0.652862 | 0.334269   |              2386 |     1.5175e+07  |
|        0.6  |  706 |  21684 |  115 | 613757 |  0.0315319  | 0.859927 | 0.0608332  |             22390 |     1.6592e+07  |
|       

In [25]:
best_threshold = 0.70

In [26]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

best_threshold = 0.70

test_scores = tabular_model.predict_proba(X_test)[:, 1]
test_pred_tuned = (test_scores >= best_threshold).astype(int)

test_precision_tuned = precision_score(y_test, test_pred_tuned, zero_division=0)
test_recall_tuned = recall_score(y_test, test_pred_tuned, zero_division=0)
test_f1_tuned = f1_score(y_test, test_pred_tuned, zero_division=0)
test_cm_tuned = confusion_matrix(y_test, test_pred_tuned)

print("Tuned threshold:", best_threshold)
print("Precision:", test_precision_tuned)
print("Recall:", test_recall_tuned)
print("F1:", test_f1_tuned)

print("\nConfusion matrix:")
print(test_cm_tuned)

print("\nClassification report:")
print(classification_report(y_test, test_pred_tuned, digits=4, zero_division=0))

Tuned threshold: 0.7
Precision: 0.060581763107488085
Recall: 0.8277541083384053
F1: 0.11290054789971775

Confusion matrix:
[[1249792   21089]
 [    283    1360]]

Classification report:
              precision    recall  f1-score   support

           0     0.9998    0.9834    0.9915   1270881
           1     0.0606    0.8278    0.1129      1643

    accuracy                         0.9832   1272524
   macro avg     0.5302    0.9056    0.5522   1272524
weighted avg     0.9986    0.9832    0.9904   1272524



In [27]:
tp = test_cm_tuned[1, 1]
fn = test_cm_tuned[1, 0]
fp = test_cm_tuned[0, 1]
tn = test_cm_tuned[0, 0]

expected_cost_test = fn * 50000 + fp * 500

print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("TN:", tn)
print("Expected cost on test set:", expected_cost_test)

TP: 1360
FP: 21089
FN: 283
TN: 1249792
Expected cost on test set: 24694500


The logistic regression model has good ranking ability, but the default threshold is not suitable for this highly imbalanced fraud problem. After threshold tuning on the validation set, the model achieved a much better balance between detecting fraud and reducing false alarms.

- default test metrics at threshold 0.50
- tuned test metrics at threshold 0.70
- confusion matrix at threshold 0.70
- expected cost at threshold 0.70

## Final tabular baseline conclusion

The logistic regression baseline showed strong fraud ranking ability, but the default threshold of 0.50 produced too many false positives.

Threshold tuning on the validation set showed that:
- the best F1-score was achieved at threshold 0.95
- the lowest expected cost was achieved at threshold 0.70

Using the cost-based threshold of 0.70 on the test set improved the fraud detection trade-off and reduced the expected cost compared with the default threshold.